In [2]:
# Import core libraries for data handling, visualization, preprocessing, modeling, and evaluation
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
import sklearn 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import LabelEncoder , MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import joblib

## Note :
Before working with model building we have to work on data prepocessing file  
Once done with preprocessing we will able to load those four data set which we can see in next cell

# Model Building

In [3]:
#  load those four datasets (X_train, X_test, y_train, y_test)
X_train=pd.read_csv(r"D:\Inno Batch 485\Machine Learning\ML Projects\Used Car  Price Prediction\Data Set\X_train.csv")
X_test=pd.read_csv(r"D:\Inno Batch 485\Machine Learning\ML Projects\Used Car  Price Prediction\Data Set\X_test.csv")
y_train= pd.read_csv(r"D:\Inno Batch 485\Machine Learning\ML Projects\Used Car  Price Prediction\Data Set\y_train.csv")
y_test= pd.read_csv(r"D:\Inno Batch 485\Machine Learning\ML Projects\Used Car  Price Prediction\Data Set\y_test.csv")

In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(620818, 15)
(155205, 15)
(620818, 1)
(155205, 1)


In [5]:
X_train.head(2)

,engine_type,fuel_type,maximum_seating,transmission,wheel_system,body_type,is_new,mileage_log,torque__log,RPM_log,hp_power_log,fuel_tank_volume_log,wheelbase_log,year_log,model_name_encoded
0,0.575758,0.8,0.230769,0.0,0.00,0.50,0.0,0.847994,0.539902,0.824721,0.513203,0.460497,0.568881,0.950472,0.005960
1,0.090909,0.8,0.230769,0.0,0.75,0.25,0.0,0.941590,0.329680,0.743936,0.252577,0.179505,0.336622,0.851268,0.005622


In [6]:
X_test.head(2)

,engine_type,fuel_type,maximum_seating,transmission,wheel_system,body_type,is_new,mileage_log,torque__log,RPM_log,hp_power_log,fuel_tank_volume_log,wheelbase_log,year_log,model_name_encoded
0,0.121212,0.8,0.230769,0.0,0.75,0.750,0.0,0.971548,0.239963,0.868233,0.269670,0.239289,0.389682,0.677187,0.005309
1,0.121212,0.8,0.230769,0.0,0.50,0.625,1.0,0.170225,0.531655,0.818065,0.488435,0.320387,0.385692,0.950472,0.004770


In [7]:
y_train.head(2)

,price
0,29489.0
1,14495.0


In [8]:
# Train a Random Forest Regressor with limited depth/leaf size to control memory usage and overfitting
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,        
    min_samples_leaf=5,   
    n_jobs=2,             
    random_state=42
)
model.fit(X_train, y_train)

C:\Users\dp431\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:1152: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(max_depth=15, min_samples_leaf=5, n_jobs=2,
                      random_state=42)

In [9]:
joblib.dump(model, 'car_price_model_random_forest_regressor.pkl')

['car_price_model_random_forest_regressor.pkl']

In [10]:
# Generate predictions on the test set using the trained Random Forest model
y_pred = model.predict(X_test)

In [11]:
# View the predicted values
y_pred

array([ 5377.83327298, 30604.86922018, 42938.09670918, ...,
       56630.05744345, 47400.926479  , 19320.4396131 ])

In [12]:
# Evaluate Random Forest performance using MAE, RMSE, and R²
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

MAE:  2379.75
RMSE: 5022.98
R²:   0.9202


In [13]:
# Check which features the Random Forest model considered most important
importances = pd.Series(model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False))

model_name_encoded      0.604433
mileage_log             0.156434
year_log                0.125031
hp_power_log            0.041251
torque__log             0.026194
is_new                  0.008370
wheelbase_log           0.007372
fuel_tank_volume_log    0.007253
maximum_seating         0.006648
RPM_log                 0.005783
engine_type             0.004207
wheel_system            0.003400
body_type               0.001994
fuel_type               0.000918
transmission            0.000713
dtype: float64


In [14]:
# Train a Linear Regression baseline model for comparison
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [15]:
lr_pred = lr.predict(X_test)
print("Linear R²:", r2_score(y_test, lr_pred))

Linear R²: 0.7645699536514202


In [16]:
joblib.dump(lr, 'car_price_model_LinearRegression.pkl')

['car_price_model_LinearRegression.pkl']

In [17]:
# Diagnostic checks: verify target ranges, feature ranges, prediction ranges, and multicollinearity
# 1. Check for extreme outliers still in your data
print("y_train range:", y_train.min(), y_train.max())
print("y_test range:", y_test.min(), y_test.max())

# 2. Check X_train for any huge/extreme values post-scaling
print(X_train.describe().T[['min','max']])

# 3. Check predictions
print("y_pred range (linear):", lr_pred.min(), lr_pred.max())

# 4. Check for dummy trap - shape check
print("X_train shape:", X_train.shape)
print("Any perfectly correlated columns?")
import numpy as np
corr_matrix = X_train.corr().abs()
high_corr = np.where((corr_matrix > 0.99) & (corr_matrix < 1.0))
print(list(zip(X_train.columns[high_corr[0]], X_train.columns[high_corr[1]])))

y_train range: price    349.0
dtype: float64 price    3195000.0
dtype: float64
y_test range: price    249.0
dtype: float64 price    1244996.0
dtype: float64
                      min  max
engine_type           0.0  1.0
fuel_type             0.0  1.0
maximum_seating       0.0  1.0
transmission          0.0  1.0
wheel_system          0.0  1.0
body_type             0.0  1.0
is_new                0.0  1.0
mileage_log           0.0  1.0
torque__log           0.0  1.0
RPM_log               0.0  1.0
hp_power_log          0.0  1.0
fuel_tank_volume_log  0.0  1.0
wheelbase_log         0.0  1.0
year_log              0.0  1.0
model_name_encoded    0.0  1.0
y_pred range (linear): -28909.28868898823 345558.1104119838
X_train shape: (620818, 15)
Any perfectly correlated columns?
[]


In [18]:
# Train an XGBoost Regressor (memory-efficient, handles large data well) and evaluate its performance
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,          
    colsample_bytree=0.8,   
    random_state=42,
    n_jobs=-1,
    tree_method='hist'      
)

xgb_model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [19]:
joblib.dump(xgb_model, 'car_price_model_XGB_Regressor.pkl')

['car_price_model_XGB_Regressor.pkl']

In [20]:
random_forest= joblib.load(r"D:\Inno Batch 485\Machine Learning\ML Projects\Used Car  Price Prediction\pkl_models\car_price_model_random_forest_regressor.pkl")

In [21]:
linear_regression= joblib.load(r"D:\Inno Batch 485\Machine Learning\ML Projects\Used Car  Price Prediction\pkl_models\car_price_model_LinearRegression.pkl")

In [22]:
xg_boost= joblib.load(r"D:\Inno Batch 485\Machine Learning\ML Projects\Used Car  Price Prediction\pkl_models\car_price_model_XGB_Regressor.pkl")

In [23]:
y_pred_random_forest = random_forest.predict(X_test)
y_pred_linear = linear_regression.predict(X_test)
y_pred_xgb = xg_boost.predict(X_test)

# Model evaluation 

In [24]:
# Reusable evaluation function reporting Accuracy(100-MAPE), MAE, RMSE, R², and MAPE for any model's predictions
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

def evaluate_model(y_test, y_pred, model_name="Model"):
    print(f"--- {model_name} ---")
    print(f"Accuracy : {100-(mean_absolute_percentage_error(y_test, y_pred)*100) :.2f}")
    print(f"MAE      : {mean_absolute_error(y_test, y_pred):.2f}")
    print(f"RMSE     : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
    print(f"R²       : {r2_score(y_test, y_pred):.4f}")
    print(f"MAPE     : {mean_absolute_percentage_error(y_test, y_pred)*100:.2f}%")
    print()

evaluate_model(y_test, y_pred_random_forest, "Random Forest")
evaluate_model(y_test, y_pred_linear, "Linear Regression")
evaluate_model(y_test, y_pred_xgb, "XgBoost Regresser")

--- Random Forest ---
Accuracy : 90.64
MAE      : 2379.75
RMSE     : 5022.98
R²       : 0.9202
MAPE     : 9.36%

--- Linear Regression ---
Accuracy : 72.75
MAE      : 5192.00
RMSE     : 8628.60
R²       : 0.7646
MAPE     : 27.25%

--- XgBoost Regresser ---
Accuracy : 91.00
MAE      : 2284.84
RMSE     : 4343.78
R²       : 0.9403
MAPE     : 9.00%

